# 02 — Exploratory Data Analysis (EDA)

**Project:** Sigmoid Bike-Sharing Case Study  
**Objective:** Understand when, how, and under what conditions people rent bikes.

This notebook builds on Notebook 01. It explores overall demand, hourly patterns, working-day effects, customer segments, weather, seasons, and monthly trends.

**Important analytical choices**
- `cnt` represents completed rentals, not unmet demand.
- We use the hourly dataset for time-of-day analysis.
- We use complete 24-hour days when calculating daily proportions from hourly records.
- We retain unusual demand observations unless there is independent evidence that they are errors.
- Commute windows are defined here as **07:00–09:00 and 17:00–19:00** (hours 7, 8, 9, 17, 18, 19).

## 1. Import libraries

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")

## 2. Load the datasets

Run this notebook independently. It locates the project folder using the same error-resistant approach as Notebook 01.

In [ ]:
def find_project_folder():
    search_locations = [Path.cwd(), *Path.cwd().parents]

    for location in search_locations:
        possible_folders = [
            location,
            location / "Sigmoid_Bike_Sharing_Case_Study"
        ]

        for folder in possible_folders:
            if (
                (folder / "data" / "hour.csv").is_file()
                and (folder / "data" / "day.csv").is_file()
            ):
                return folder

    common_locations = [
        Path.home() / "Documents" / "Sigmoid_Bike_Sharing_Case_Study",
        Path.home() / "Downloads" / "Sigmoid_Bike_Sharing_Case_Study",
        Path.home() / "Desktop" / "Sigmoid_Bike_Sharing_Case_Study"
    ]

    for folder in common_locations:
        if (
            (folder / "data" / "hour.csv").is_file()
            and (folder / "data" / "day.csv").is_file()
        ):
            return folder

    try:
        import tkinter as tk
        from tkinter import filedialog

        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)

        selected_folder = filedialog.askdirectory(
            title="Select the Sigmoid_Bike_Sharing_Case_Study folder"
        )
        root.destroy()

        if selected_folder:
            folder = Path(selected_folder)
            if (
                (folder / "data" / "hour.csv").is_file()
                and (folder / "data" / "day.csv").is_file()
            ):
                return folder

    except Exception as error:
        print("Folder picker could not be opened:", error)

    raise FileNotFoundError(
        "Could not locate data/hour.csv and data/day.csv. "
        "Select the main project folder."
    )

In [ ]:
BASE_DIR = find_project_folder()
DATA_DIR = BASE_DIR / "data"

hour_path = DATA_DIR / "hour.csv"
day_path = DATA_DIR / "day.csv"

hour_df = pd.read_csv(hour_path, parse_dates=["dteday"])
day_df = pd.read_csv(day_path, parse_dates=["dteday"])

print("Project folder:", BASE_DIR)
print("Hourly dataset:", hour_df.shape)
print("Daily dataset:", day_df.shape)

## 3. Prepare analysis columns

In [ ]:
hour_work = hour_df.copy()
day_work = day_df.copy()

hour_work["weekday_name"] = hour_work["dteday"].dt.day_name()
hour_work["month_name"] = hour_work["dteday"].dt.month_name()

hour_work["day_type"] = np.where(
    hour_work["workingday"] == 1,
    "Working day",
    "Non-working day"
)

day_work["weekday_name"] = day_work["dteday"].dt.day_name()
day_work["month_name"] = day_work["dteday"].dt.month_name()

print("Analysis datasets prepared.")

In [ ]:
# Map hour numbers to readable day parts.
def get_daypart(hour):
    if 5 <= hour <= 11:
        return "Morning"
    elif 12 <= hour <= 16:
        return "Afternoon"
    elif 17 <= hour <= 20:
        return "Evening"
    else:
        return "Night"

hour_work["daypart"] = hour_work["hr"].apply(get_daypart)

daypart_order = ["Morning", "Afternoon", "Evening", "Night"]

hour_work["daypart"] = pd.Categorical(
    hour_work["daypart"],
    categories=daypart_order,
    ordered=True
)

hour_work["is_weekend"] = (
    hour_work["weekday"].isin([0, 6])
).astype(int)

commute_hours = [7, 8, 9, 17, 18, 19]

hour_work["is_commute_hour"] = (
    hour_work["hr"].isin(commute_hours)
).astype(int)

display(
    hour_work[
        ["dteday", "hr", "weekday_name", "day_type",
         "daypart", "is_weekend", "is_commute_hour", "cnt"]
    ].head()
)

## 4. Overall demand overview

In [ ]:
overall_metrics = pd.Series({
    "Total rentals": hour_work["cnt"].sum(),
    "Average hourly rentals": hour_work["cnt"].mean(),
    "Median hourly rentals": hour_work["cnt"].median(),
    "Maximum hourly rentals": hour_work["cnt"].max(),
    "Minimum hourly rentals": hour_work["cnt"].min(),
    "Number of hourly records": len(hour_work),
    "Number of days": hour_work["dteday"].nunique()
})

display(overall_metrics.to_frame("value"))

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(hour_work["cnt"], bins=40, kde=True)
plt.title("Distribution of Hourly Bike Rentals")
plt.xlabel("Number of Rentals per Hour")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(x=hour_work["cnt"])
plt.title("Hourly Rental Count — Boxplot")
plt.xlabel("Number of Rentals per Hour")
plt.tight_layout()
plt.show()

## 5. Hourly demand pattern

In [ ]:
hourly_demand = (
    hour_work.groupby("hr", as_index=False)["cnt"]
    .mean()
    .rename(columns={"cnt": "average_rentals"})
)

display(hourly_demand)

In [ ]:
plt.figure(figsize=(12, 5))
sns.lineplot(
    data=hourly_demand,
    x="hr",
    y="average_rentals",
    marker="o"
)
plt.title("Average Bike Rentals by Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Average Rentals")
plt.xticks(range(24))
plt.tight_layout()
plt.show()

In [ ]:
peak_hours = (
    hourly_demand.sort_values(
        "average_rentals",
        ascending=False
    )
)

display(peak_hours.head(10))

**Interpretation prompts:** Which hours have the highest average demand? Are peaks concentrated in the morning, evening, or both? What operational decisions could be informed by these peaks?

## 6. Working-day vs non-working-day demand

In [ ]:
daytype_hourly = (
    hour_work.groupby(
        ["hr", "day_type"],
        observed=False,
        as_index=False
    )["cnt"]
    .mean()
)

display(daytype_hourly.head(10))

In [ ]:
plt.figure(figsize=(12, 6))
sns.lineplot(
    data=daytype_hourly,
    x="hr",
    y="cnt",
    hue="day_type",
    marker="o"
)
plt.title("Hourly Demand: Working Days vs Non-working Days")
plt.xlabel("Hour of Day")
plt.ylabel("Average Rentals")
plt.xticks(range(24))
plt.legend(title="Day Type")
plt.tight_layout()
plt.show()

In [ ]:
daytype_summary = (
    hour_work.groupby("day_type")["cnt"]
    .agg(["mean", "median", "sum", "count"])
)

display(daytype_summary)

## 7. Weekday demand pattern

In [ ]:
weekday_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

weekday_demand = (
    hour_work.groupby("weekday_name")["cnt"]
    .mean()
    .reindex(weekday_order)
    .rename("average_rentals")
    .reset_index()
)

display(weekday_demand)

In [ ]:
plt.figure(figsize=(11, 5))
sns.barplot(
    data=weekday_demand,
    x="weekday_name",
    y="average_rentals"
)
plt.title("Average Bike Rentals by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Average Hourly Rentals")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 8. Day-part demand

In [ ]:
daypart_demand = (
    hour_work.groupby(
        ["daypart", "day_type"],
        observed=False,
        as_index=False
    )["cnt"]
    .mean()
)

display(daypart_demand)

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(
    data=daypart_demand,
    x="daypart",
    y="cnt",
    hue="day_type",
    order=daypart_order
)
plt.title("Average Demand by Day Part and Day Type")
plt.xlabel("Day Part")
plt.ylabel("Average Rentals per Hour")
plt.tight_layout()
plt.show()

## 9. Customer segmentation: casual vs registered

In [ ]:
customer_totals = pd.DataFrame({
    "Customer Type": ["Casual", "Registered"],
    "Total Rentals": [
        hour_work["casual"].sum(),
        hour_work["registered"].sum()
    ]
})

customer_totals["Share_Percentage"] = (
    customer_totals["Total Rentals"]
    / customer_totals["Total Rentals"].sum()
    * 100
)

display(customer_totals)

In [ ]:
plt.figure(figsize=(7, 5))
sns.barplot(
    data=customer_totals,
    x="Customer Type",
    y="Total Rentals"
)
plt.title("Total Rentals by Customer Type")
plt.xlabel("Customer Type")
plt.ylabel("Total Rentals")
plt.tight_layout()
plt.show()

In [ ]:
customer_hourly = (
    hour_work.groupby("hr", as_index=False)[["casual", "registered"]]
    .mean()
    .melt(
        id_vars="hr",
        var_name="customer_type",
        value_name="average_rentals"
    )
)

display(customer_hourly.head())

In [ ]:
plt.figure(figsize=(12, 5))
sns.lineplot(
    data=customer_hourly,
    x="hr",
    y="average_rentals",
    hue="customer_type",
    marker="o"
)
plt.title("Hourly Demand by Customer Type")
plt.xlabel("Hour of Day")
plt.ylabel("Average Rentals")
plt.xticks(range(24))
plt.tight_layout()
plt.show()

In [ ]:
customer_daytype = (
    hour_work.groupby("day_type")[["casual", "registered"]]
    .sum()
)

customer_daytype["casual_share_pct"] = (
    customer_daytype["casual"]
    / (customer_daytype["casual"] + customer_daytype["registered"])
    * 100
)

customer_daytype["registered_share_pct"] = (
    customer_daytype["registered"]
    / (customer_daytype["casual"] + customer_daytype["registered"])
    * 100
)

display(customer_daytype)

## 10. Weather and demand

In [ ]:
weather_labels = {
    1: "Clear / Few clouds",
    2: "Mist / Cloudy",
    3: "Light rain / Snow",
    4: "Heavy rain / Snow"
}

hour_work["weather_label"] = hour_work["weathersit"].map(weather_labels)
day_work["weather_label"] = day_work["weathersit"].map(weather_labels)

weather_demand = (
    hour_work.groupby("weather_label", as_index=False)["cnt"]
    .agg(["mean", "median", "count"])
)

display(weather_demand)

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(
    data=weather_demand.reset_index(),
    x="weather_label",
    y="mean"
)
plt.title("Average Hourly Rentals by Weather Condition")
plt.xlabel("Weather Condition")
plt.ylabel("Average Rentals")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

**Caution:** Weather category 4 has very few observations. Its average is unstable and should not be compared with the other categories as if all groups had similar sample sizes.

## 11. Seasonal demand

In [ ]:
season_labels = {
    1: "Season 1",
    2: "Season 2",
    3: "Season 3",
    4: "Season 4"
}

hour_work["season_label"] = hour_work["season"].map(season_labels)
day_work["season_label"] = day_work["season"].map(season_labels)

season_demand = (
    hour_work.groupby("season_label", as_index=False)["cnt"]
    .agg(["mean", "median", "count"])
)

display(season_demand)

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(
    data=hour_work,
    x="season_label",
    y="cnt",
    order=["Season 1", "Season 2", "Season 3", "Season 4"]
)
plt.title("Hourly Rental Distribution by Season Code")
plt.xlabel("Season Code")
plt.ylabel("Hourly Rentals")
plt.tight_layout()
plt.show()

**Important:** The x-axis deliberately uses numeric season codes, not names such as spring or winter. Notebook 01 identified that the dataset's season labels need to be verified against its README. Only replace these labels after confirming the documentation.

## 12. Monthly demand

In [ ]:
monthly_demand = (
    day_work.groupby("mnth", as_index=False)["cnt"]
    .mean()
    .rename(columns={"cnt": "average_daily_rentals"})
)

display(monthly_demand)

In [ ]:
plt.figure(figsize=(11, 5))
sns.lineplot(
    data=monthly_demand,
    x="mnth",
    y="average_daily_rentals",
    marker="o"
)
plt.title("Average Daily Rentals by Month")
plt.xlabel("Month")
plt.ylabel("Average Daily Rentals")
plt.xticks(range(1, 13))
plt.tight_layout()
plt.show()

## 13. Year-over-year demand

In [ ]:
yearly_demand = (
    day_work.groupby("yr")["cnt"]
    .agg(["mean", "sum", "count"])
)

display(yearly_demand)

In [ ]:
plt.figure(figsize=(7, 5))
sns.barplot(
    data=day_work,
    x="yr",
    y="cnt",
    errorbar=None
)
plt.title("Average Daily Rentals by Year Code")
plt.xlabel("Year Code (0 = 2011, 1 = 2012)")
plt.ylabel("Average Daily Rentals")
plt.tight_layout()
plt.show()

## 14. Commute-window demand concentration

We calculate each date's share of rentals occurring during the selected commute windows. Only complete 24-hour days are used so that the denominator is comparable across dates.

In [ ]:
complete_hour_work = hour_work[
    hour_work["dteday"].isin(
        hour_work.groupby("dteday")["hr"].nunique()
        .loc[lambda values: values == 24].index
    )
].copy()

daily_commute = (
    complete_hour_work.groupby(
        ["dteday", "workingday", "day_type"],
        as_index=False
    )
    .agg(
        daily_rentals=("cnt", "sum"),
        commute_rentals=(
            "cnt",
            lambda values: values[
                complete_hour_work.loc[values.index, "is_commute_hour"] == 1
            ].sum()
        )
    )
)

daily_commute["commute_share_pct"] = (
    daily_commute["commute_rentals"]
    / daily_commute["daily_rentals"]
    * 100
)

display(daily_commute.head())

In [ ]:
commute_summary = (
    daily_commute.groupby("day_type")["commute_share_pct"]
    .agg(["mean", "median", "count"])
)

display(commute_summary)

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=daily_commute,
    x="day_type",
    y="commute_share_pct"
)
plt.title("Daily Commute-Window Share by Day Type")
plt.xlabel("Day Type")
plt.ylabel("Commute Rentals as % of Daily Rentals")
plt.tight_layout()
plt.show()

## 15. Correlation between numeric variables

In [ ]:
correlation_cols = [
    "temp", "atemp", "hum", "windspeed",
    "casual", "registered", "cnt"
]

correlation_matrix = hour_work[correlation_cols].corr()

display(correlation_matrix)

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)
plt.title("Correlation Matrix of Numeric Variables")
plt.tight_layout()
plt.show()

**Interpretation caution:** Correlation does not establish causation. Also, `cnt` is exactly the sum of `casual` and `registered`, so their correlations with total rentals are mechanically high.

## 16. Key observations summary

In [ ]:
eda_summary = pd.Series({
    "Total observed rentals": int(hour_work["cnt"].sum()),
    "Average hourly rentals": round(hour_work["cnt"].mean(), 2),
    "Peak average demand hour": int(
        hourly_demand.loc[hourly_demand["average_rentals"].idxmax(), "hr"]
    ),
    "Highest average hourly demand": round(
        hourly_demand["average_rentals"].max(), 2
    ),
    "Customer segment with more rentals": (
        "Registered"
        if hour_work["registered"].sum() > hour_work["casual"].sum()
        else "Casual"
    ),
    "Commute-window hours used": str(commute_hours)
})

display(eda_summary.to_frame("result"))

## 17. Business interpretation and limitations

Use the outputs above to write evidence-based observations. Do not copy expected numbers from another project without checking your own results.

Questions to answer:
1. At what hours is demand highest?
2. How does the hourly profile differ between working and non-working days?
3. Do casual and registered customers show different usage patterns?
4. How does observed demand vary with weather?
5. Which season codes and months have higher demand?
6. What proportion of daily rentals occurs in the selected commute windows?

**Business limitation:** This dataset records completed rentals. It does not include station-level inventory, failed rental attempts, or origin-destination data. EDA can identify when completed rentals are concentrated, but cannot establish where riders failed to find bikes.

### Next notebook

**`03_hypothesis_testing.ipynb`** — formal statistical testing of selected business hypotheses, including effect sizes and confidence intervals.